# 4. Statistics

This notebook computes every statistical test in the preprint paper:

- Mann–Whitney U tests for determining whether Search Interest differs between groups of prefectures;
- Spearman correlations between Search Interest and distance from Fukushima.

Fukushima Prefecture is included only in the 2004–2010 Mann–Whitney U tests.
From 2011 on, it is excluded from the Mann–Whitney U tests because its Search Interest is the maximum of 100 in
every period, far above every other prefecture.
It is also excluded from all Spearman correlations because its distance from itself is zero.

In [1]:
from pathlib import Path
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr

DATA = Path("../data")
prefectures = pd.read_csv(DATA / "prefectures.csv")
d = prefectures[prefectures["prefecture"] != "Fukushima"]   # Fukushima excluded

## Mann–Whitney U test

Each test is two-sided and compares a comparison group with a baseline group.
The probability of higher interest is defined as

$$\frac{U}{n_\mathrm{comparison} \times n_\mathrm{baseline}},$$

where $U$ is the Mann–Whitney U statistic of the comparison group.
The probability of higher interest is the chance that a randomly chosen comparison prefecture has higher Search
Interest than a randomly chosen baseline prefecture, with ties counted as one half.
In the tables, n is the number of prefectures in each group and the p-value is two-sided.

In [2]:
def mann_whitney(comparison, baseline):
    result = mannwhitneyu(comparison, baseline, alternative="two-sided")
    return {"n (baseline)": len(baseline), "n (comparison)": len(comparison), "p-value": result.pvalue,
            "probability of higher interest": result.statistic / (len(comparison) * len(baseline))}

def show(rows):
    return pd.DataFrame(rows).style.format({"p-value": "{:.2e}", "probability of higher interest": "{:.3f}"}).hide(axis="index")

### Supplementary Table S1: the effect of host and border status on Search Interest, 2004–2010

In [3]:
pre = prefectures.assign(group=prefectures["host_2004-2010"].map({1: "Host", 0: "No-host"}) + " / "
                         + prefectures["border_2004-2010"].map({1: "Border", 0: "No-border"}))
values = {g: rows["search_interest_2004-2010"] for g, rows in pre.groupby("group")}
table_s1 = [{"variable": variable, "baseline": baseline, "comparison": comparison,
             **mann_whitney(values[comparison], values[baseline])}
            for variable, baseline, comparison in [
                ("Host status", "No-host / Border", "Host / Border"),
                ("Host status", "No-host / No-border", "Host / No-border"),
                ("Border status", "Host / No-border", "Host / Border"),
                ("Border status", "No-host / No-border", "No-host / Border")]]
show(table_s1)

variable,baseline,comparison,n (baseline),n (comparison),p-value,probability of higher interest
Host status,No-host / Border,Host / Border,25,6,3.41e-04,0.980
Host status,No-host / No-border,Host / No-border,9,7,1.27e-02,0.881
Border status,Host / No-border,Host / Border,7,6,6.68e-01,0.583
Border status,No-host / No-border,No-host / Border,9,25,1.42e-01,0.669


### The effect of host status on Search Interest in 2011, Fukushima excluded

In [4]:
test_2011 = {"baseline": "No-host", "comparison": "Host",
             **mann_whitney(d.loc[d["host_2011"] == 1, "search_interest_2011"], d.loc[d["host_2011"] == 0, "search_interest_2011"])}
show([test_2011])

baseline,comparison,n (baseline),n (comparison),p-value,probability of higher interest
No-host,Host,34,12,4.75e-01,0.571


### Supplementary Table S2: the effect of host status on Search Interest, 2012–2024, Fukushima excluded

In [5]:
POST = [("2012-2014", "Shutdown (2012–2014)"), ("2015-2019", "Initial restart (2015–2019)"), ("2020-2024", "Recent (2020–2024)")]
table_s2 = [{"period": label, "baseline": "No-host", "comparison": "Host",
             **mann_whitney(d.loc[d[f"host_{p}"] == 1, f"search_interest_{p}"], d.loc[d[f"host_{p}"] == 0, f"search_interest_{p}"])}
            for p, label in POST]
show(table_s2)

period,baseline,comparison,n (baseline),n (comparison),p-value,probability of higher interest
Shutdown (2012–2014),No-host,Host,34,12,2.65e-04,0.857
Initial restart (2015–2019),No-host,Host,34,12,1.44e-05,0.925
Recent (2020–2024),No-host,Host,34,12,3.16e-06,0.956


## Spearman correlation

Non-parametric, two-sided Spearman correlation tests are performed between Search Interest and distance from
Fukushima for 2004–2010, 2011 and 2012–2014, with Fukushima excluded (46 prefectures).
In the table, n is the number of prefectures and ρ is Spearman's rank correlation coefficient, which runs from −1
to 1.

In [6]:
spearman = []
for period in ["2004-2010", "2011", "2012-2014"]:
    r = spearmanr(d["distance_km"], d[f"search_interest_{period}"])
    spearman.append({"period": period.replace("-", "–"), "n": len(d), "ρ": r.statistic, "p-value": r.pvalue})
pd.DataFrame(spearman).style.format({"ρ": "{:.2f}", "p-value": "{:.3g}"}).hide(axis="index")

period,n,ρ,p-value
2004–2010,46,0.17,0.253
2011,46,-0.54,9.08e-05
2012–2014,46,-0.14,0.356
